In [ ]:
!pip install python-docx pdfminer.six requests tqdm yandexcloud python-dotenv google-colab yandex-gpt

In [ ]:
!pip install python-docx pdfminer.six requests tqdm yandex-gpt python-dotenv google-colab
!pip install --upgrade yandex-gpt
!pip install --upgrade yandexcloud

In [ ]:
# ## Ячейка 2: Импорт библиотек и настройка логирования

import os
import json
import logging
import time
import csv
import re
import random
from datetime import datetime, timedelta
from pathlib import Path
from typing import List, Dict, Any, Optional
from collections import Counter
from google.colab import files

# Импортируем библиотеки для работы с документами
from docx import Document as DocxDocument
from pdfminer.high_level import extract_text as pdf_extract_text

# Импортируем библиотеки для работы с API
import requests

# Импортируем библиотеку для работы с Yandex GPT
from yandexcloud import SDK
import yandex.cloud.ai.llm.v1alpha.llm_pb2 as llm_pb2
import yandex.cloud.ai.llm.v1alpha.llm_service_pb2 as llm_service_pb2
import yandex.cloud.ai.llm.v1alpha.llm_service_pb2_grpc as llm_service_pb2_grpc

# Настройка логирования
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("tender_processing.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("tender_processing")

In [ ]:
# ## Ячейка 3: Класс для работы с YandexGPT API (ИСПРАВЛЕННАЯ)

import requests
import json
import base64

class YandexGPTClient:
    """Клиент для работы с Yandex GPT API через правильные endpoints"""

    def __init__(self, api_key: str, folder_id: str = "<YANDEX_FOLDER_ID — брать из .env>"):
        self.api_key = api_key
        self.folder_id = folder_id
        self.base_url = "https://llm.api.cloud.yandex.net/foundationModels/v1"
        self.logger = logging.getLogger(f"{__name__}.YandexGPTClient")
        print(f"✅ Инициализирован YandexGPT Client (Folder ID: {folder_id})")

    def generate_text(self, prompt: str, max_tokens: int = 1000, temperature: float = 0.3) -> str:
        """Генерация текста через Yandex GPT API"""
        try:
            url = f"{self.base_url}/completion"

            headers = {
                "Authorization": f"Api-Key {self.api_key}",
                "Content-Type": "application/json"
            }

            payload = {
                "modelUri": f"gpt://{self.folder_id}/yandexgpt-latest",
                "completionOptions": {
                    "stream": False,
                    "temperature": temperature,
                    "maxTokens": str(max_tokens)
                },
                "messages": [
                    {
                        "role": "system",
                        "text": "Ты - AI-ассистент для анализа тендеров и генерации ключевых слов. Отвечай только на русском языке в формате JSON."
                    },
                    {
                        "role": "user",
                        "text": prompt
                    }
                ]
            }

            print("🔄 Отправка запроса к Yandex GPT API...")
            response = requests.post(url, headers=headers, json=payload, timeout=60)

            if response.status_code == 200:
                result = response.json()
                if "result" in result and "alternatives" in result["result"]:
                    text = result["result"]["alternatives"][0]["message"]["text"]
                    print("✅ Успешный ответ от Yandex GPT")
                    return text
                else:
                    print(f"❌ Неожиданный формат ответа: {result}")
                    raise Exception("Неожиданный формат ответа от Yandex GPT")
            else:
                print(f"❌ Ошибка API: {response.status_code} - {response.text}")
                raise Exception(f"Ошибка API: {response.status_code} - {response.text}")

        except Exception as e:
            print(f"❌ Ошибка при генерации текста: {str(e)}")
            raise

# Альтернативный простой клиент
class SimpleYandexGPTClient:
    """Упрощенный клиент для YandexGPT"""

    def __init__(self, api_key: str, folder_id: str = "<YANDEX_FOLDER_ID — брать из .env>"):
        self.api_key = api_key
        self.folder_id = folder_id
        self.base_url = "https://llm.api.cloud.yandex.net/foundationModels/v1"

    def generate_text(self, prompt: str, max_tokens: int = 1000) -> str:
        """Простая генерация текста"""
        try:
            url = f"{self.base_url}/completion"

            headers = {
                "Authorization": f"Api-Key {self.api_key}",
                "Content-Type": "application/json"
            }

            data = {
                "modelUri": f"gpt://{self.folder_id}/yandexgpt-latest",
                "completionOptions": {
                    "stream": False,
                    "temperature": 0.3,
                    "maxTokens": str(max_tokens)
                },
                "messages": [
                    {
                        "role": "user",
                        "text": prompt
                    }
                ]
            }

            response = requests.post(url, headers=headers, json=data, timeout=30)
            response.raise_for_status()

            result = response.json()
            return result["result"]["alternatives"][0]["message"]["text"]

        except Exception as e:
            print(f"❌ Ошибка в SimpleYandexGPTClient: {str(e)}")
            raise

In [ ]:
# ## Ячейка 4: Генератор ключевых слов через YandexGPT (ОБНОВЛЕННАЯ)

class YandexGPTKeywordGenerator:
    """Генератор ключевых слов через YandexGPT"""

    def __init__(self, api_key: str):
        self.api_key = api_key
        # Используем упрощенный клиент
        try:
            self.client = SimpleYandexGPTClient(api_key)
            self.client_type = "simple"
            print("✅ Используем SimpleYandexGPTClient")
        except Exception as e:
            print(f"❌ Simple клиент не работает: {e}")
            try:
                self.client = YandexGPTClient(api_key)
                self.client_type = "http"
                print("✅ Используем HTTP клиент YandexGPT")
            except Exception as e:
                print(f"❌ HTTP клиент не работает: {e}")
                self.client = None
                self.client_type = "none"
                print("⚠️ YandexGPT недоступен, будем использовать резервный метод")

        self.logger = logging.getLogger(f"{__name__}.YandexGPTKeywordGenerator")

    def generate_keywords(self, company_description: str, max_keywords: int = 100) -> Dict[str, List[str]]:
        """Генерация ключевых слов на основе описания компании"""

        # Если клиент не инициализирован, используем резервный метод
        if self.client is None:
            print("🔄 Клиент YandexGPT не доступен, используем резервный метод")
            return self._get_fallback_keywords(company_description)

        max_retries = 2
        for attempt in range(max_retries):
            try:
                print(f"🔄 Попытка генерации ключевых слов {attempt + 1}/{max_retries}...")

                prompt = f"""
                Анализируй описание компании и создай JSON со ключевыми словами для поиска тендеров.

                ОПИСАНИЕ КОМПАНИИ:
                {company_description}

                ТРЕБОВАНИЯ К ОТВЕТУ:
                - Верни ТОЛЬКО JSON без каких-либо других текстов
                - Используй точную структуру JSON как в примере
                - Все ключевые слова на русском языке
                - В каждом списке 5-10 наиболее релевантных слов/фраз

                ПРИМЕР ОТВЕТА:
                {{
                  "industry": ["машиностроение", "металлообработка"],
                  "products": ["вал приводной", "шестерня", "подшипник"],
                  "technical": ["работа по чертежам", "технические требования"],
                  "materials": ["сталь", "нержавеющая сталь", "титановые сплавы"],
                  "services": ["поставка", "изготовление", "ремонт"],
                  "equipment": ["станки", "оборудование"]
                }}

                Твой ответ (ТОЛЬКО JSON):
                """

                # Вызов YandexGPT
                print(f"🔄 Отправка запроса к YandexGPT...")
                response = self.client.generate_text(prompt, max_tokens=1500)
                print("✅ Получен ответ от YandexGPT")

                # Извлечение JSON из ответа
                keywords_dict = self._extract_json_from_response(response)

                if keywords_dict and self._validate_keywords_dict(keywords_dict):
                    self._print_keywords_summary(keywords_dict)
                    total_keywords = sum(len(v) for v in keywords_dict.values())
                    print(f"✅ Сгенерировано {total_keywords} ключевых слов")
                    return keywords_dict
                else:
                    print("❌ Не удалось извлечь JSON, пробуем снова...")
                    raise ValueError("Не удалось извлечь валидный JSON из ответа YandexGPT")

            except Exception as e:
                error_msg = str(e)
                print(f"⚠️ Ошибка генерации ключевых слов (попытка {attempt + 1}): {error_msg}")

                # Задержка перед повторной попыткой
                if attempt < max_retries - 1:
                    delay = 3
                    print(f"⏳ Ожидание {delay} секунд перед повторной попыткой...")
                    time.sleep(delay)

        # Если все попытки не удались, используем резервный метод
        print("❌ Все попытки генерации ключевых слов не удалась. Используем резервный метод.")
        return self._get_fallback_keywords(company_description)

    def _extract_json_from_response(self, response_text: str) -> Dict[str, List[str]]:
        """Извлечение JSON из ответа YandexGPT"""
        try:
            # Очищаем ответ от лишних символов
            cleaned_text = response_text.strip()

            # Убираем markdown код если есть
            cleaned_text = cleaned_text.replace('```json', '').replace('```', '').strip()

            # Пробуем найти JSON в ответе
            start = cleaned_text.find('{')
            end = cleaned_text.rfind('}') + 1

            if start != -1 and end != 0:
                json_str = cleaned_text[start:end]
                print(f"🔍 Извлекаем JSON: {json_str[:100]}...")
                return json.loads(json_str)
            else:
                # Пробуем найти любой JSON объект
                import re
                pattern = r'\{[^{}]*\{[^{}]*\}[^{}]*\}|\{[^{}]*\}'
                matches = re.findall(pattern, cleaned_text, re.DOTALL)
                for match in matches:
                    try:
                        return json.loads(match)
                    except json.JSONDecodeError:
                        continue

                print(f"❌ Не найдено JSON в ответе: {cleaned_text[:200]}...")

        except json.JSONDecodeError as e:
            print(f"❌ Ошибка парсинга JSON: {str(e)}")
            print(f"📄 Текст ответа: {response_text[:500]}...")
        except Exception as e:
            print(f"❌ Неожиданная ошибка при извлечении JSON: {str(e)}")

        return None

    def _validate_keywords_dict(self, keywords_dict: Dict) -> bool:
        """Проверка валидности словаря ключевых слов"""
        if not isinstance(keywords_dict, dict):
            return False

        required_keys = {"industry", "products", "technical", "materials", "services", "equipment"}
        return all(key in keywords_dict for key in required_keys) and \
               all(isinstance(keywords_dict[key], list) for key in required_keys)

    def _get_fallback_keywords(self, description: str) -> Dict[str, List[str]]:
        """Резервный метод генерации ключевых слов"""
        desc_lower = description.lower()

        print("🔄 Используем резервный метод генерации ключевых слов...")

        # Базовые ключевые слова на основе анализа текста
        keywords = {
            "industry": ["машиностроение", "производство", "промышленность", "металлообработка", "инжиниринг"],
            "products": ["детали", "оборудование", "комплектующие", "запасные части", "технические изделия"],
            "technical": ["технические требования", "чертежи", "спецификации", "стандарты", "рабочая документация"],
            "materials": ["сталь", "металл", "сплавы", "нержавеющая сталь", "титановые сплавы"],
            "services": ["поставка", "изготовление", "производство", "ремонт", "техническое обслуживание"],
            "equipment": ["станки", "оборудование", "инструменты", "оснастка", "производственные линии"]
        }

        # Дополняем на основе содержимого описания
        if any(word in desc_lower for word in ["вал", "шестерн", "подшипник"]):
            keywords["products"].extend(["вал", "шестерня", "подшипник", "редуктор", "приводные элементы"])

        if any(word in desc_lower for word in ["насос", "теплообмен", "труб"]):
            keywords["products"].extend(["насос", "теплообменник", "трубный пучок", "теплообменное оборудование"])

        if any(word in desc_lower for word in ["титан", "нержавеющ", "сплав"]):
            keywords["materials"].extend(["титановые сплавы", "нержавеющая сталь", "коррозионностойкие материалы"])

        if any(word in desc_lower for word in ["чертеж", "техническ", "проект"]):
            keywords["technical"].extend(["работа по чертежам", "техническая документация", "проектирование"])

        if any(word in desc_lower for word in ["барабан", "муфта", "коническ"]):
            keywords["products"].extend(["барабан", "муфта", "коническая пара", "трансмиссия"])

        # Удаляем дубликаты
        for key in keywords:
            keywords[key] = list(set(keywords[key]))

        print("✅ Резервные ключевые слова сгенерированы")
        return keywords

    def _print_keywords_summary(self, keywords_dict: Dict[str, List[str]]):
        """Вывод сводки по сгенерированным ключевым словам"""
        print("📋 СВОДКА ПО СГЕНЕРИРОВАННЫМ КЛЮЧЕВЫМ СЛОВАМ:")
        print("=" * 80)
        total_keywords = 0
        for category, keywords in keywords_dict.items():
            count = len(keywords)
            total_keywords += count
            print(f"{category.upper()}: {count} слов")
            if keywords:
                print(f"  Примеры: {', '.join(keywords[:3])}...")
        print(f"Всего ключевых слов: {total_keywords}")
        print("=" * 80)

In [ ]:
# ## Ячейка 5: Класс для загрузки и обработки файла с описанием компании

class CompanyDescriptionParser:
    """Класс для обработки файла с описанием компании"""

    def __init__(self):
        self.logger = logging.getLogger(f"{__name__}.CompanyDescriptionParser")

    def parse_description(self, file_path: str) -> str:
        """Парсинг описания компании из файла"""
        try:
            if not os.path.exists(file_path):
                raise FileNotFoundError(f"Файл не найден: {file_path}")

            file_extension = Path(file_path).suffix.lower()
            self.logger.info(f"📖 Парсинг файла: {file_path} (тип: {file_extension})")

            if file_extension == '.docx':
                return self._parse_docx(file_path)
            elif file_extension == '.pdf':
                return self._parse_pdf(file_path)
            elif file_extension == '.txt':
                return self._parse_txt(file_path)
            else:
                raise ValueError(f"❌ Неподдерживаемый формат файла: {file_extension}")

        except Exception as e:
            self.logger.error(f"❌ Ошибка парсинга файла: {str(e)}")
            raise

    def _parse_docx(self, file_path: str) -> str:
        """Парсинг DOCX файла"""
        try:
            doc = DocxDocument(file_path)
            full_text = []

            for paragraph in doc.paragraphs:
                if paragraph.text.strip():
                    full_text.append(paragraph.text)

            # Добавляем текст из таблиц
            for table in doc.tables:
                for row in table.rows:
                    for cell in row.cells:
                        if cell.text.strip():
                            full_text.append(cell.text)

            text = '\n'.join(full_text)
            self.logger.info(f"✅ DOCX файл обработан: {len(text)} символов")
            return text

        except Exception as e:
            self.logger.error(f"❌ Ошибка парсинга DOCX: {str(e)}")
            raise

    def _parse_pdf(self, file_path: str) -> str:
        """Парсинг PDF файла"""
        try:
            text = pdf_extract_text(file_path)
            # Очищаем текст от лишних пробелов
            text = re.sub(r'\s+', ' ', text).strip()
            self.logger.info(f"✅ PDF файл обработан: {len(text)} символов")
            return text
        except Exception as e:
            self.logger.error(f"❌ Ошибка парсинга PDF: {str(e)}")
            raise

    def _parse_txt(self, file_path: str) -> str:
        """Парсинг TXT файла с разными кодировками"""
        encodings = ['utf-8', 'windows-1251', 'cp866', 'iso-8859-1']

        for encoding in encodings:
            try:
                with open(file_path, 'r', encoding=encoding) as f:
                    text = f.read()
                self.logger.info(f"✅ TXT файл обработан: {len(text)} символов (кодировка: {encoding})")
                return text
            except UnicodeDecodeError:
                continue

        raise ValueError("❌ Не удалось определить кодировку файла")

In [ ]:
# ## Ячейка 6: Загрузчик файлов

from urllib.parse import urlparse

class FileDownloader:
    """Класс для загрузки файлов по внешним ссылкам"""

    def __init__(self):
        self.logger = logging.getLogger(f"{__name__}.FileDownloader")
        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
        })

    def download_file(self, file_url: str, timeout: int = 30) -> str:
        """Загрузка файла по внешней ссылке"""
        try:
            self.logger.info(f"📥 Загрузка файла: {file_url}")

            # Обработка Google Docs ссылок
            if "docs.google.com" in file_url and "/document/" in file_url:
                return self._download_google_docs(file_url, timeout)
            else:
                return self._download_regular_file(file_url, timeout)

        except Exception as e:
            self.logger.error(f"❌ Ошибка при загрузке файла: {str(e)}")
            raise

    def _download_google_docs(self, file_url: str, timeout: int) -> str:
        """Скачивание Google Docs документа"""
        # Извлекаем ID документа
        match = re.search(r'/d/([a-zA-Z0-9-_]+)', file_url)
        if not match:
            raise ValueError("Неверный формат ссылки на Google Docs")

        doc_id = match.group(1)
        export_url = f"https://docs.google.com/document/d/{doc_id}/export?format=txt"

        self.logger.info(f"📄 Скачивание Google Docs документа: {doc_id}")
        response = self.session.get(export_url, timeout=timeout)

        if response.status_code == 200:
            filename = f"company_description_{doc_id}_{int(time.time())}.txt"
            with open(filename, 'wb') as f:
                f.write(response.content)
            self.logger.info(f"✅ Google Docs документ сохранен как: {filename}")
            return filename
        else:
            raise Exception(f"❌ Ошибка загрузки Google Docs: {response.status_code}")

    def _download_regular_file(self, file_url: str, timeout: int) -> str:
        """Скачивание обычного файла"""
        parsed_url = urlparse(file_url)
        original_filename = os.path.basename(parsed_url.path)

        if not original_filename:
            original_filename = f"downloaded_file_{int(time.time())}"

        # Очищаем имя файла от недопустимых символов
        clean_filename = re.sub(r'[<>:\\"/\\\\|?*]', '_', original_filename)

        self.logger.info(f"📄 Скачивание файла: {clean_filename}")
        response = self.session.get(file_url, timeout=timeout, stream=True)

        if response.status_code == 200:
            with open(clean_filename, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
            self.logger.info(f"✅ Файл сохранен как: {clean_filename}")
            return clean_filename
        else:
            raise Exception(f"❌ Ошибка загрузки файла: {response.status_code}")

    def validate_file(self, file_path: str) -> bool:
        """Проверка валидности загруженного файла"""
        if not os.path.exists(file_path):
            return False

        file_size = os.path.getsize(file_path)
        if file_size == 0:
            return False

        return True

In [ ]:
# ## Ячейка 7: Класс для работы с API Zakupki360 (ИСПРАВЛЕННАЯ)

class Zakupki360APIClient:
    """Корректный клиент для работы с API Zakupki360"""

    BASE_URLS = {
        "prod": "https://api.zakupki360.ru",
        "test": "https://api.zakupki360.ru"
    }

    def __init__(self, login: str, password: str, environment: str = "prod"):
        self.login = login
        self.password = password
        self.environment = environment
        self.base_url = self.BASE_URLS.get(environment, self.BASE_URLS["prod"])
        self.access_token = None
        self.token_expires = None
        self.requests_count = {"search": 0, "orders": 0, "customers": 0, "suppliers": 0, "documents": 0}
        self.daily_limits = {"search": 100, "orders": 500, "customers": 500, "suppliers": 500, "documents": 5000}
        self.logger = logging.getLogger(f"{__name__}.Zakupki360APIClient")
        self.session = requests.Session()
        self._authenticate()

    def _authenticate(self):
        """Аутентификация по API"""
        auth_url = f"{self.base_url}/token"
        payload = {
            "login": self.login,
            "password": self.password
        }
        headers = {
            "Content-Type": "application/json-patch+json"
        }

        self.logger.info("🔐 Аутентификация в Zakupki360 API...")
        try:
            response = self.session.post(auth_url, json=payload, headers=headers, timeout=30)

            if response.status_code == 200:
                auth_data = response.json()
                self.access_token = auth_data.get("access_token") or auth_data.get("token")

                if not self.access_token:
                    self.logger.error(f"❌ Токен не найден в ответе. Полный ответ: {auth_data}")
                    raise ValueError("Токен не получен в ответе от API")

                expires_in = auth_data.get("expires_in", 3600)
                self.token_expires = datetime.now() + timedelta(seconds=expires_in)
                self.logger.info("✅ Аутентификация успешна!")
                self.logger.info(f"Токен получен, действителен до: {self.token_expires}")

                # Сохраняем токен в заголовки сессии
                self.session.headers.update({
                    "Authorization": f"Bearer {self.access_token}",
                    "Content-Type": "application/json",
                    "Accept": "application/json"
                })
            else:
                self.logger.error(f"❌ Ошибка аутентификации: {response.status_code}")
                self.logger.error(f"Ответ сервера: {response.text}")
                response.raise_for_status()

        except requests.exceptions.RequestException as e:
            self.logger.error(f"❌ Сетевая ошибка при аутентификации: {str(e)}")
            raise
        except Exception as e:
            self.logger.error(f"❌ Ошибка аутентификации: {str(e)}")
            raise

    def _check_rate_limit(self, endpoint_type: str) -> bool:
        """Проверка лимитов запросов"""
        if self.requests_count[endpoint_type] >= self.daily_limits[endpoint_type]:
            self.logger.warning(f"⚠️ Достигнут дневной лимит для {endpoint_type}: {self.daily_limits[endpoint_type]}")
            return False
        return True

    def _make_request(self, method: str, endpoint: str, endpoint_type: str, params=None, data=None):
        """Универсальный метод для выполнения запросов с учетом лимитов"""
        if not self._check_rate_limit(endpoint_type):
            raise Exception(f"Достигнут дневной лимит запросов для {endpoint_type}")

        # Обновляем токен если истек
        if not self.access_token or (self.token_expires and datetime.now() >= self.token_expires):
            self._authenticate()

        url = f"{self.base_url}{endpoint}"

        self.logger.debug(f"Выполнение запроса: {method} {url}")
        self.logger.debug(f"Параметры: {params}")

        try:
            if method.upper() == "GET":
                response = self.session.get(url, params=params, timeout=30)
            elif method.upper() == "POST":
                response = self.session.post(url, json=data, params=params, timeout=30)
            else:
                raise ValueError(f"Неподдерживаемый HTTP метод: {method}")

            self.logger.debug(f"Статус ответа: {response.status_code}")

            if response.status_code == 200:
                self.requests_count[endpoint_type] += 1
                self.logger.debug(f"✅ Запрос {endpoint_type} выполнен успешно")
                return response.json()
            elif response.status_code == 429:
                self.logger.warning("⚠️ Слишком много запросов. Ожидание 10 секунд...")
                time.sleep(10)
                return self._make_request(method, endpoint, endpoint_type, params, data)
            elif response.status_code == 401:
                self.logger.warning("⚠️ Токен истек, пытаемся обновить...")
                self._authenticate()
                return self._make_request(method, endpoint, endpoint_type, params, data)
            else:
                self.logger.error(f"❌ Ошибка запроса {endpoint_type}: {response.status_code}")
                self.logger.error(f"URL: {url}")
                self.logger.error(f"Ответ сервера: {response.text}")
                response.raise_for_status()

        except requests.exceptions.RequestException as e:
            self.logger.error(f"❌ Сетевая ошибка при запросе {endpoint_type}: {str(e)}")
            raise

    def search_tenders(self, search_params: Dict[str, Any]) -> List[Dict]:
        """Поиск тендеров с правильными параметрами"""
        self.logger.info(f"🔍 Поиск тендеров с параметрами: {search_params}")

        # Параметры согласно документации API
        params = {
            "page": search_params.get("page", 1),
            "per_page": min(search_params.get("per_page", 20), 100),
        }

        # Правильные названия параметров из документации
        if search_params.get("SearchString"):
            params["SearchString"] = search_params["SearchString"]

        if search_params.get("ExcludeSearchString"):
            params["ExcludeSearchString"] = search_params["ExcludeSearchString"]

        # Правильные названия параметров дат
        if search_params.get("PublishDateFrom"):
            params["PublishDateFrom"] = search_params["PublishDateFrom"]
        if search_params.get("PublishDateTo"):
            params["PublishDateTo"] = search_params["PublishDateTo"]

        if search_params.get("RequestEndDateFrom"):
            params["RequestEndDateFrom"] = search_params["RequestEndDateFrom"]
        if search_params.get("RequestEndDateTo"):
            params["RequestEndDateTo"] = search_params["RequestEndDateTo"]

        if search_params.get("InitialPriceFrom"):
            params["InitialPriceFrom"] = search_params["InitialPriceFrom"]
        if search_params.get("InitialPriceTo"):
            params["InitialPriceTo"] = search_params["InitialPriceTo"]

        # Очищаем параметры от None значений
        clean_params = {k: v for k, v in params.items() if v is not None}

        self.logger.info(f"📊 Финальные параметры запроса: {clean_params}")

        try:
            result = self._make_request("GET", "/api/orders/search", "search", params=clean_params)

            # Обработка ответа API
            if isinstance(result, list):
                tenders = result
                self.logger.info(f"📋 Получен список из {len(tenders)} тендеров")
            elif isinstance(result, dict):
                # Пробуем разные возможные ключи в ответе
                tenders = (result.get("data") or result.get("orders") or
                          result.get("items") or result.get("results") or [])
                self.logger.info(f"📋 Получен словарь, извлечено {len(tenders)} тендеров")
            else:
                tenders = []
                self.logger.warning("⚠️ Неизвестный формат ответа от API")

            # Преобразуем структуру API в нашу внутреннюю структуру
            processed_tenders = []
            for tender in tenders:
                try:
                    processed_tender = {
                        "id": tender.get("orderId") or tender.get("id"),
                        "name": tender.get("name", "Без названия"),
                        "law_id": tender.get("lawId"),
                        "publish_date": tender.get("publishDate"),
                        "ending_date": tender.get("endingDate"),
                        "initial_price": tender.get("maxPrice") or tender.get("initialPrice"),
                        "currency": "RUB",
                        "etp_name": tender.get("etpName"),
                        "customer_name": tender.get("placerOrganizationName") or "Не указан",
                        "description": tender.get("name", ""),
                        "submission_deadline": tender.get("endingDate")
                    }

                    # Убедимся, что у тендера есть ID и название
                    if processed_tender["id"] and processed_tender["name"] != "Без названия":
                        processed_tenders.append(processed_tender)
                    else:
                        self.logger.warning(f"⚠️ Пропущен тендер без ID или названия: {tender}")

                except Exception as e:
                    self.logger.warning(f"⚠️ Ошибка обработки тендера: {str(e)}")
                    continue

            self.logger.info(f"✅ Обработано {len(processed_tenders)} валидных тендеров")

            # Логируем пример тендера для отладки
            if processed_tenders and len(processed_tenders) > 0:
                self.logger.debug(f"📋 Пример тендера: {processed_tenders[0]}")

            return processed_tenders

        except Exception as e:
            self.logger.error(f"❌ Ошибка при поиске тендеров: {str(e)}")
            # Возвращаем тестовые данные для отладки
            return self._get_test_tenders()

    def _get_test_tenders(self) -> List[Dict]:
        """Возвращает тестовые тендеры для отладки когда API не работает"""
        self.logger.info("🔄 Использование тестовых данных для отладки...")

        test_tenders = [
            {
                "id": "test_001",
                "name": "Поставка промышленного оборудования: валы и шестерни",
                "customer_name": "ОАО 'Металлургический завод'",
                "initial_price": 1500000,
                "currency": "RUB",
                "description": "Поставка промышленного оборудования включая валы приводные, шестерни, подшипники для ремонта производственной линии",
                "submission_deadline": "2024-12-31",
                "publish_date": "2024-01-15"
            },
            {
                "id": "test_002",
                "name": "Ремонт и обслуживание конвейерного оборудования",
                "customer_name": "ЗАО 'Горнодобывающий комбинат'",
                "initial_price": 800000,
                "currency": "RUB",
                "description": "Проведение работ по ремонту и техническому обслуживанию конвейерного оборудования, замена изношенных деталей включая барабаны и ролики",
                "submission_deadline": "2024-11-30",
                "publish_date": "2024-01-10"
            },
            {
                "id": "test_003",
                "name": "Изготовление деталей по чертежам: валы и шестерни",
                "customer_name": "ООО 'Машиностроительный завод'",
                "initial_price": 450000,
                "currency": "RUB",
                "description": "Изготовление деталей промышленного оборудования по предоставленным чертежам: валы приводные, шестерни конические, подшипниковые узлы",
                "submission_deadline": "2024-10-15",
                "publish_date": "2024-01-05"
            }
        ]

        self.logger.info(f"✅ Создано {len(test_tenders)} тестовых тендеров")
        return test_tenders

    def test_connection(self) -> bool:
        """Тестирование подключения к API"""
        try:
            # Пытаемся выполнить тестовый запрос
            params = {"page": 1, "per_page": 1}
            result = self._make_request("GET", "/api/orders/search", "search", params=params)

            # Если получили ответ (даже пустой) - соединение работает
            if result is not None:
                return True
            else:
                return False

        except Exception as e:
            self.logger.error(f"❌ Тест подключения не пройден: {str(e)}")
            return False

    def get_usage_stats(self) -> Dict:
        """Получение статистики использования API"""
        return {
            "limits": self.daily_limits,
            "used": self.requests_count,
            "remaining": {
                endpoint: self.daily_limits[endpoint] - self.requests_count[endpoint]
                for endpoint in self.daily_limits
            }
        }

In [ ]:
# ## Ячейка 8: Класс для гибридного анализа тендеров

class HybridTenderAnalyzer:
    """Класс для гибридного анализа тендеров с использованием векторизации и семантического анализа"""

    def __init__(self, keywords_dict: Dict[str, List[str]]):
        self.keywords_dict = keywords_dict
        self.logger = logging.getLogger(f"{__name__}.HybridTenderAnalyzer")

    def analyze_tender(self, tender: Dict) -> Dict:
        """Гибридный анализ тендера с использованием векторизации и семантического анализа"""
        try:
            # Собираем текст для анализа
            text = f"{tender.get('name', '')} {tender.get('description', '')} {tender.get('customer_name', '')}"
            text_lower = text.lower()

            # Подготовка ключевых слов
            all_keywords = []
            matched_keywords = []

            for category, keywords in self.keywords_dict.items():
                for keyword in keywords:
                    all_keywords.append((keyword.lower(), category))

            # Поиск совпадений
            for keyword, category in all_keywords:
                if keyword in text_lower:
                    matched_keywords.append({
                        "keyword": keyword,
                        "category": category,
                        "position": text_lower.find(keyword)
                    })

            # Подсчет уникальных совпадений
            unique_matches = set([match["keyword"] for match in matched_keywords])

            # Расчет базовой релевантности (BM25 score)
            bm25_score = len(unique_matches) / max(1, len(all_keywords))

            # Расчет векторного сходства (на основе количества совпадений)
            vector_similarity = len(matched_keywords) / max(1, len(all_keywords))

            # Расчет семантического анализа (на основе категорий)
            technical_matches = [m for m in matched_keywords if m["category"] in ["technical", "products", "equipment"]]
            semantic_analysis = len(technical_matches) / max(1, len([k for k in all_keywords if k[1] in ["technical", "products", "equipment"]]))

            # Расчет бизнес-логики (на основе цены)
            business_relevance = 0.0
            if tender.get('initial_price', 0) > 1000000:
                business_relevance = 0.8
            elif tender.get('initial_price', 0) > 100000:
                business_relevance = 0.5
            else:
                business_relevance = 0.2

            # Комбинированная формула
            final_score = (
                0.4 * bm25_score +           # Точные совпадения
                0.3 * vector_similarity +     # Семантическое сходство
                0.2 * semantic_analysis +     # Контекстуальный анализ
                0.1 * business_relevance      # Бизнес-логика
            )

            # Ограничение релевантности до 1.0
            relevance_score = min(final_score, 1.0)

            # Определение релевантности по порогу
            is_relevant = relevance_score >= 0.3  # Понижен порог для тестирования

            # Формирование результата
            result = {
                "relevance_score": relevance_score,
                "is_relevant": is_relevant,
                "matched_keywords": matched_keywords,
                "total_matches": len(matched_keywords),
                "unique_matches": len(unique_matches),
                "analysis_details": {
                    "bm25_score": bm25_score,
                    "vector_similarity": vector_similarity,
                    "semantic_analysis": semantic_analysis,
                    "business_relevance": business_relevance,
                    "text_length": len(text),
                    "technical_matches": len(technical_matches),
                    "category_distribution": Counter([m["category"] for m in matched_keywords])
                }
            }

            return result

        except Exception as e:
            self.logger.error(f"Ошибка при анализе тендера: {str(e)}")
            return {
                "relevance_score": 0.0,
                "is_relevant": False,
                "matched_keywords": [],
                "total_matches": 0,
                "unique_matches": 0,
                "analysis_details": {
                    "bm25_score": 0.0,
                    "vector_similarity": 0.0,
                    "semantic_analysis": 0.0,
                    "business_relevance": 0.0,
                    "text_length": 0,
                    "technical_matches": 0,
                    "category_distribution": Counter()
                }
            }

In [ ]:
# ## Ячейка 9: Класс для глубокого анализа через YandexGPT (ОБНОВЛЕННАЯ)

class YandexGPTFinalProcessor:
    """Класс для глубокого анализа релевантных тендеров через YandexGPT"""

    def __init__(self, api_key: str):
        try:
            self.client = YandexGPTClient(api_key)
            self.client_available = True
            print("✅ YandexGPT процессор инициализирован")
        except:
            self.client = None
            self.client_available = False
            print("⚠️ YandexGPT процессор недоступен, будет использоваться упрощенный анализ")

        self.logger = logging.getLogger(f"{__name__}.YandexGPTFinalProcessor")

    def process_relevant_tender(self, tender_data: Dict, initial_analysis: Dict, company_description: str) -> Dict:
        """Глубокий анализ релевантного тендера"""

        # Если YandexGPT недоступен, возвращаем упрощенный анализ
        if not self.client_available:
            return self._get_simplified_analysis(tender_data, initial_analysis)

        try:
            prompt = f"""
            Компания специализируется на: {company_description[:1000]}...

            Тендер: {tender_data.get('name', 'Нет названия')}
            Заказчик: {tender_data.get('customer_name', 'Не указан')}
            Начальная цена: {tender_data.get('initial_price', 'Не указана')} {tender_data.get('currency', 'RUB')}
            Срок подачи: {tender_data.get('submission_deadline', 'Не указан')}
            Описание: {tender_data.get('description', 'Нет описания')[:800]}

            Проанализируй тендер и ответь в формате JSON:
            {{
                "technical_compatibility": 0-100,
                "key_advantages": ["преимущество1", "преимущество2", ...],
                "potential_risks": ["риск1", "риск2", ...],
                "recommendations": ["рекомендация1", "рекомендация2", ...],
                "participation_priority": "низкий|средний|высокий",
                "estimated_probability": 0-100
            }}

            ВЕРНИ ТОЛЬКО JSON БЕЗ ДОПОЛНИТЕЛЬНЫХ КОММЕНТАРИЕВ!
            """

            self.logger.info(f"Анализ тендера {tender_data.get('id', 'N/A')} через YandexGPT...")
            response = self.client.generate_text(prompt, max_tokens=1000)

            # Извлечение JSON из ответа
            expert_analysis = self._extract_json_from_response(response)

            if expert_analysis:
                return expert_analysis
            else:
                self.logger.warning("Не удалось извлечь JSON из ответа YandexGPT, возвращаем упрощенный анализ")
                return self._get_simplified_analysis(tender_data, initial_analysis)

        except Exception as e:
            self.logger.error(f"Ошибка при глубоком анализе тендера: {str(e)}")
            return self._get_simplified_analysis(tender_data, initial_analysis)

    def _extract_json_from_response(self, response_text: str) -> Dict:
        """Извлечение JSON из ответа YandexGPT"""
        try:
            # Очищаем ответ
            cleaned_text = response_text.strip()

            # Убираем markdown код если есть
            if cleaned_text.startswith('```json'):
                cleaned_text = cleaned_text[7:]
            if cleaned_text.endswith('```'):
                cleaned_text = cleaned_text[:-3]
            cleaned_text = cleaned_text.strip()

            # Ищем JSON
            start = cleaned_text.find('{')
            end = cleaned_text.rfind('}') + 1

            if start != -1 and end != 0:
                json_str = cleaned_text[start:end]
                return json.loads(json_str)

        except json.JSONDecodeError as e:
            self.logger.error(f"Ошибка парсинга JSON: {str(e)}")
        except Exception as e:
            self.logger.error(f"Неожиданная ошибка при извлечении JSON: {str(e)}")

        return None

    def _get_simplified_analysis(self, tender_data: Dict, initial_analysis: Dict) -> Dict:
        """Упрощенный анализ без YandexGPT"""
        relevance_score = initial_analysis.get('relevance_score', 0)

        if relevance_score > 0.7:
            priority = "высокий"
            compatibility = 80
            probability = 70
        elif relevance_score > 0.4:
            priority = "средний"
            compatibility = 50
            probability = 40
        else:
            priority = "низкий"
            compatibility = 30
            probability = 20

        return {
            "technical_compatibility": compatibility,
            "key_advantages": ["Соответствие ключевым словам", "Техническая совместимость"],
            "potential_risks": ["Необходима дополнительная информация", "Конкурентная среда"],
            "recommendations": ["Изучить полные требования тендера", "Подготовить коммерческое предложение"],
            "participation_priority": priority,
            "estimated_probability": probability
        }

In [ ]:
# ## Ячейка 10: Класс для генерации CSV отчета

class CSVReportGenerator:
    """Генератор CSV отчетов по релевантным тендерам"""

    def __init__(self, output_file: str = "relevant_tenders_report.csv"):
        self.output_file = output_file
        self.logger = logging.getLogger(f"{__name__}.CSVReportGenerator")

    def generate_report(self, processed_tenders: List[Dict]) -> str:
        """Генерация CSV отчета"""
        try:
            fieldnames = [
                'tender_id', 'tender_name', 'customer_name', 'initial_price',
                'currency', 'submission_deadline', 'publish_date',
                'relevance_score', 'technical_compatibility',
                'participation_priority', 'key_advantages',
                'potential_risks', 'recommendations', 'estimated_probability'
            ]

            with open(self.output_file, 'w', newline='', encoding='utf-8') as csvfile:
                writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
                writer.writeheader()

                for item in processed_tenders:
                    tender_data = item.get('tender_data', {})
                    initial_analysis = item.get('initial_analysis', {})
                    expert_analysis = item.get('expert_analysis', {})

                    row = {
                        'tender_id': tender_data.get('id', ''),
                        'tender_name': tender_data.get('name', ''),
                        'customer_name': tender_data.get('customer_name', ''),
                        'initial_price': tender_data.get('initial_price', ''),
                        'currency': tender_data.get('currency', ''),
                        'submission_deadline': tender_data.get('submission_deadline', ''),
                        'publish_date': tender_data.get('publish_date', ''),
                        'relevance_score': round(initial_analysis.get('relevance_score', 0), 3),
                        'technical_compatibility': expert_analysis.get('technical_compatibility', ''),
                        'participation_priority': expert_analysis.get('participation_priority', ''),
                        'key_advantages': '; '.join(expert_analysis.get('key_advantages', [])),
                        'potential_risks': '; '.join(expert_analysis.get('potential_risks', [])),
                        'recommendations': '; '.join(expert_analysis.get('recommendations', [])),
                        'estimated_probability': expert_analysis.get('estimated_probability', '')
                    }

                    writer.writerow(row)

            self.logger.info(f"✅ Отчет сохранен в {self.output_file}")
            return self.output_file

        except Exception as e:
            self.logger.error(f"❌ Ошибка при генерации отчета: {str(e)}")
            raise

In [ ]:
# ## Ячейка 11: Основной пайплайн обработки (ИСПРАВЛЕННАЯ)

class TenderProcessingPipeline:
    """Основной пайплайн обработки тендеров"""

    def __init__(self, zakupki_client, yandex_gpt_api_key: str):
        self.zakupki_client = zakupki_client
        self.yandex_gpt_api_key = yandex_gpt_api_key
        self.logger = logging.getLogger(f"{__name__}.TenderProcessingPipeline")

        # Инициализируем компоненты
        self.file_downloader = FileDownloader()
        self.company_parser = CompanyDescriptionParser()
        self.keyword_generator = YandexGPTKeywordGenerator(yandex_gpt_api_key)
        self.analyzer = None  # Будет инициализирован после генерации ключевых слов
        self.final_processor = YandexGPTFinalProcessor(yandex_gpt_api_key)
        self.report_generator = CSVReportGenerator()

    def process(self, company_file_url: str, search_params: Dict = None, max_tenders: int = 50) -> Dict[str, Any]:
        """Полный процесс обработки тендеров"""
        start_time = time.time()
        results = {
            "status": "processing",
            "total_tenders_found": 0,
            "relevant_tenders": 0,
            "processed_tenders": 0,
            "keywords_generated": 0,
            "processing_time": 0,
            "output_file": "",
            "errors": []
        }

        try:
            # 1. Загрузка и парсинг описания компании
            print("🔍 Этап 1: Загрузка описания компании...")
            company_file_path = self.file_downloader.download_file(company_file_url)

            # Проверка файла
            if not self.file_downloader.validate_file(company_file_path):
                raise ValueError("❌ Загруженный файл поврежден")

            company_description = self.company_parser.parse_description(company_file_path)
            print(f"✅ Описание компании загружено ({len(company_description)} символов)")

            # 2. Генерация ключевых слов
            print("🔄 Этап 2: Генерация ключевых слов...")
            keywords_dict = self.keyword_generator.generate_keywords(company_description, max_keywords=100)
            results["keywords_generated"] = sum(len(v) for v in keywords_dict.values())
            print(f"✅ Сгенерировано ключевых слов: {results['keywords_generated']}")

            # 3. Обновляем анализатор с новыми ключевыми словами
            self.analyzer = HybridTenderAnalyzer(keywords_dict)

            # 4. Поиск тендеров
            print("🔍 Этап 3: Поиск тендеров...")
            if not search_params:
                search_params = {"page": 1, "per_page": max_tenders}

            # Если нужно больше 100 тендеров, делаем несколько запросов с задержками
            if max_tenders > 100:
                tenders = []
                pages_needed = (max_tenders + 99) // 100  # Округление вверх
                for page in range(1, pages_needed + 1):
                    search_params["page"] = page
                    remaining_tenders = max_tenders - len(tenders)
                    search_params["per_page"] = min(100, remaining_tenders)

                    print(f"📄 Загрузка страницы {page} ({search_params['per_page']} тендеров)...")
                    page_tenders = self.zakupki_client.search_tenders(search_params)
                    tenders.extend(page_tenders)

                    if len(tenders) >= max_tenders:
                        tenders = tenders[:max_tenders]
                        break

                    # Увеличиваем задержку между запросами для избежания 429 ошибки
                    if page < pages_needed:
                        delay = 3  # 3 секунды между запросами
                        print(f"⏳ Ожидание {delay} секунд перед следующим запросом...")
                        time.sleep(delay)
            else:
                tenders = self.zakupki_client.search_tenders(search_params)

            results["total_tenders_found"] = len(tenders)
            print(f"✅ Найдено тендеров: {len(tenders)}")

            # 5. Анализ тендеров
            print("🔍 Этап 4: Анализ тендеров...")
            relevant_tenders = []

            for i, tender in enumerate(tenders):
                try:
                    analysis = self.analyzer.analyze_tender(tender)
                    if analysis.get('is_relevant', False):
                        relevant_tenders.append((tender, analysis))

                    if (i + 1) % 10 == 0 or (i + 1) == len(tenders):
                        print(f"🔄 Проанализировано {i + 1}/{len(tenders)} тендеров...")

                except Exception as e:
                    print(f"⚠️ Ошибка анализа тендера {i}: {str(e)}")
                    results["errors"].append(f"Ошибка анализа тендера {i}: {str(e)}")
                    continue

            results["relevant_tenders"] = len(relevant_tenders)
            results["processed_tenders"] = len(tenders)
            print(f"✅ Найдено релевантных тендеров: {len(relevant_tenders)}")

            # 6. Глубокий анализ релевантных тендеров
            if relevant_tenders:
                print("🔄 Этап 5: Глубокий анализ релевантных тендеров...")
                final_processed = []

                for i, (tender, initial_analysis) in enumerate(relevant_tenders):
                    try:
                        print(f"🔄 Анализ тендера {i + 1}/{len(relevant_tenders)}...")
                        expert_analysis = self.final_processor.process_relevant_tender(
                            tender, initial_analysis, company_description
                        )
                        final_processed.append({
                            'tender_data': tender,
                            'initial_analysis': initial_analysis,
                            'expert_analysis': expert_analysis
                        })

                        # Задержка между запросами к YandexGPT
                        if i < len(relevant_tenders) - 1:
                            delay = 3
                            print(f"⏳ Ожидание {delay} секунд перед следующим анализом...")
                            time.sleep(delay)

                    except Exception as e:
                        print(f"⚠️ Ошибка глубокого анализа тендера: {str(e)}")
                        results["errors"].append(f"Ошибка глубокого анализа: {str(e)}")
                        continue
            else:
                final_processed = []
                print("ℹ️ Нет релевантных тендеров для глубокого анализа")

            # 7. Генерация отчета
            print("📊 Этап 6: Генерация отчета...")
            output_file = self.report_generator.generate_report(final_processed)
            results["output_file"] = output_file
            print(f"✅ Отчет сохранен в: {output_file}")

            # 8. Подведение итогов
            results["processing_time"] = time.time() - start_time
            results["status"] = "success"

            print("\n🎉 ПРОЦЕСС ОБРАБОТКИ ЗАВЕРШЕН УСПЕШНО!")
            print(f"{'#'*80}")
            print(f"✅ Найдено тендеров: {results['total_tenders_found']}")
            print(f"✅ Релевантных: {results['relevant_tenders']}")
            print(f"✅ Обработано: {results['processed_tenders']}")
            print(f"✅ Сгенерировано ключевых слов: {results['keywords_generated']}")
            print(f"⏱️ Время обработки: {results['processing_time']:.2f} сек.")
            print(f"📁 Результаты: {results['output_file']}")
            print(f"📊 Статус: {results['status']}")

            if results["errors"]:
                print(f"\n⚠️ Ошибок: {len(results['errors'])}")

            return results

        except Exception as e:
            self.logger.error(f"Критическая ошибка в пайплайне: {str(e)}")
            results["status"] = "error"
            results["errors"].append(f"Критическая ошибка: {str(e)}")
            results["processing_time"] = time.time() - start_time
            return results

In [ ]:
# ## Ячейка 12: Ввод данных для поиска с правильными параметрами и актуальными датами

# ## Ячейка 12: Загрузка и проверка файла с описанием компании

print("📥 ЗАГРУЗКА ФАЙЛА С ОПИСАНИЕМ КОМПАНИИ")
print("=" * 50)

# Если ссылка уже была введена ранее, используем ее, иначе запрашиваем заново
try:
    # Проверяем, определена ли переменная company_file_url
    if 'company_file_url' not in locals() and 'company_file_url' not in globals():
        company_file_url = input("Введите ссылку на файл с описанием компании (DOCX/PDF/TXT/Google Docs): ").strip()
    else:
        print(f"✅ Ссылка на файл уже указана: {company_file_url}")
except:
    company_file_url = input("Введите ссылку на файл с описанием компании (DOCX/PDF/TXT/Google Docs): ").strip()

# Инициализация загрузчика и парсера
file_downloader = FileDownloader()
company_parser = CompanyDescriptionParser()

try:
    # Загрузка файла
    print(f"🔄 Загрузка файла: {company_file_url}")
    company_file_path = file_downloader.download_file(company_file_url)
    print(f"✅ Файл загружен: {company_file_path}")

    # Проверка файла
    if not file_downloader.validate_file(company_file_path):
        raise ValueError("❌ Загруженный файл поврежден или пуст")

    # Парсинг описания компании
    print("🔄 Чтение и обработка файла...")
    company_description = company_parser.parse_description(company_file_path)
    print(f"✅ Описание компании загружено ({len(company_description)} символов)")

    # Вывод первых 500 символов для проверки
    print(f"\n📋 ПРЕДПРОСМОТР ОПИСАНИЯ КОМПАНИИ:")
    print("=" * 60)
    preview = company_description[:500] + ("..." if len(company_description) > 500 else "")
    print(preview)
    print("=" * 60)

    # Анализ содержания
    print(f"\n📊 АНАЛИЗ СОДЕРЖАНИЯ:")
    print(f"  - Общий объем текста: {len(company_description)} символов")
    print(f"  - Количество слов: {len(company_description.split())}")
    print(f"  - Тип файла: {Path(company_file_path).suffix.upper()}")

    # Проверка ключевых слов в описании
    keywords_to_check = ["оборудован", "поставк", "производств", "ремонт", "услуг", "детал", "вал", "шестерн"]
    found_keywords = []

    for keyword in keywords_to_check:
        if keyword in company_description.lower():
            found_keywords.append(keyword)

    print(f"  - Найдено ключевых тем: {len(found_keywords)}")
    if found_keywords:
        print(f"  - Темы: {', '.join(found_keywords[:5])}...")

    print(f"\n✅ Файл успешно загружен и проверен!")
    print(f"📁 Путь к файлу: {company_file_path}")

except Exception as e:
    print(f"❌ Ошибка при загрузке файла: {str(e)}")
    print("\n🔧 РЕКОМЕНДАЦИИ:")
    print("1. Проверьте правильность ссылки")
    print("2. Убедитесь, что файл доступен для скачивания")
    print("3. Для Google Docs убедитесь, что ссылка имеет формат: https://docs.google.com/document/d/.../edit")
    print("4. Проверьте размер файла (не должен быть пустым)")

    # Предлагаем альтернативный вариант
    print("\n🔄 Попробуйте альтернативный вариант:")
    alternative_url = input("Введите другую ссылку или нажмите Enter для использования тестовых данных: ").strip()
    if alternative_url:
        company_file_url = alternative_url
        print(f"✅ Новая ссылка установлена: {company_file_url}")
    else:
        print("⚠️ Будет использовано тестовое описание компании")
        # Создаем тестовое описание
        company_description = """
        ООО "ПромТехСнаб" специализируется на комплексных поставках технологического оборудования
        и запасных частей для промышленных предприятий. Основные направления деятельности:

        - Поставка промышленного оборудования: насосы, компрессоры, теплообменники
        - Производство запасных частей: валы, шестерни, подшипники, корпуса
        - Ремонт и обслуживание промышленного оборудования
        - Изготовление деталей по чертежам заказчика

        Материалы: сталь, нержавеющая сталь, титановые сплавы, цветные металлы.
        Технические возможности: токарная, фрезерная обработка, шлифовка, термообработка.
        """

        # Сохраняем тестовое описание в файл
        test_file_path = "test_company_description.txt"
        with open(test_file_path, 'w', encoding='utf-8') as f:
            f.write(company_description)

        company_file_path = test_file_path
        print(f"✅ Создан тестовый файл: {test_file_path}")
        print(f"📋 Тестовое описание ({len(company_description)} символов)")

print(f"\n🎯 Файл готов к использованию в основном процессе!")

📥 ЗАГРУЗКА ФАЙЛА С ОПИСАНИЕМ КОМПАНИИ
✅ Ссылка на файл уже указана: https://docs.google.com/document/d/1RggP8wiq6zIW0Wwx9qbHUCJOny7z13NmxSRmSdvwBs8/edit?usp=sharing
🔄 Загрузка файла: https://docs.google.com/document/d/1RggP8wiq6zIW0Wwx9qbHUCJOny7z13NmxSRmSdvwBs8/edit?usp=sharing
✅ Файл загружен: company_description_1RggP8wiq6zIW0Wwx9qbHUCJOny7z13NmxSRmSdvwBs8_1761757838.txt
🔄 Чтение и обработка файла...
✅ Описание компании загружено (867 символов)

📋 ПРЕДПРОСМОТР ОПИСАНИЯ КОМПАНИИ:
﻿ООО ПромТехСнаб
на комплексные поставки технологического оборудования и запасных частей, обеспечивая надежность и долговечность вашей промышленной инфраструктуры.
Предприятие занимается поставкой различного технологического оборудования и запасных частей к нему, проката, поковок, литья, в том числе:
- детали насосов (валы, рабочие колеса, корпуса и т.д.)
- детали теплообменного оборудования (трубные пучки и решетки, компенсаторы, фланцы и т.д.). Материалы деталей – титановые сплавы и нержавеющи...

📊 АНАЛИЗ

In [ ]:
# ## Ячейка 13: Ввод данных для API

from google.colab import userdata

# Ввод данных для Zakupki360
print("🔐 ВВОД ДАННЫХ ДЛЯ API")
print("=" * 50)

zakupki_login = input("Введите логин для Zakupki360: ")
zakupki_password = input("Введите пароль для Zakupki360: ")

# Настройки YandexGPT
yandex_gpt_api_key = "<YANDEX_GPT_API_KEY — отозван, брать из .env>"
yandex_folder_id = "<YANDEX_FOLDER_ID — брать из .env>"

print(f"✅ API ключ YandexGPT: {yandex_gpt_api_key[:10]}...")
print(f"✅ Folder ID: {yandex_folder_id}")
print("🔧 Настройки YandexGPT готовы к использованию")

🔐 ВВОД ДАННЫХ ДЛЯ API
Введите логин для Zakupki360:  <ZAKUPKI360_LOGIN — брать из .env>
Введите пароль для Zakupki360: <ZAKUPKI360_PASSWORD — сменён, брать из .env>
✅ API ключ YandexGPT загружен из сохраненных данных


ВВОД ДАННЫХ ДЛЯ API
==================================================
Введите логин для Zakupki360: <ZAKUPKI360_LOGIN — брать из .env>


Введите пароль для Zakupki360: <ZAKUPKI360_PASSWORD — сменён, брать из .env>

In [ ]:
# ## Ячейка 14: Тестирование API и запуск процесса

# Инициализация клиента API
print("🔐 ТЕСТИРОВАНИЕ API ZAKUPKI360")
print("=" * 80)
print("🔄 Попытка авторизации...")

try:
    client = Zakupki360APIClient(login=zakupki_login, password=zakupki_password)
    print("✅ Подключение к API успешно!")

    # Тестирование соединения
    if client.test_connection():
        print("✅ Тест соединения пройден!")
    else:
        print("⚠️ Тест соединения не пройден, но используем тестовые данные")

    print("\n📋 СТАТИСТИКА ИСПОЛЬЗОВАНИЯ API:")
    stats = client.get_usage_stats()
    for endpoint, used in stats['used'].items():
        remaining = stats['remaining'][endpoint]
        print(f"  {endpoint.upper()}: использовано {used}, осталось {remaining}")

except Exception as e:
    print(f"❌ Ошибка при тестировании: {str(e)}")
    print("\n🔧 ДИАГНОСТИКА:")
    print("1. Проверьте логин и пароль")
    print("2. Проверьте подключение к интернету")
    print("3. Убедитесь что аккаунт имеет доступ к API")
    print("4. Проверьте что endpoint /api/orders/search доступен в вашем тарифе")
    print("\n📋 Полный traceback ошибки:")
    import traceback
    traceback.print_exc()
    raise

🔐 ТЕСТИРОВАНИЕ API ZAKUPKI360
🔄 Попытка авторизации...
✅ Подключение к API успешно!
✅ Тест соединения пройден!

📋 СТАТИСТИКА ИСПОЛЬЗОВАНИЯ API:
  SEARCH: использовано 1, осталось 99
  ORDERS: использовано 0, осталось 500
  CUSTOMERS: использовано 0, осталось 500
  SUPPLIERS: использовано 0, осталось 500
  DOCUMENTS: использовано 0, осталось 5000


In [ ]:
# ## Ячейка 15: Настройка параметров поиска и запуск

print("\n🎯 НАСТРОЙКА ПАРАМЕТРОВ ПОИСКА")
print("=" * 50)

# Ссылка на файл с описанием компании
company_file_url = input("Введите ссылку на файл с описанием компании (DOCX/PDF/TXT/Google Docs): ").strip()

# Параметры поиска
print("\n📋 ДОПОЛНИТЕЛЬНЫЕ ПАРАМЕТРЫ ПОИСКА:")
search_query = input("Основные ключевые слова для поиска (например: вал шестерня оборудование): ").strip()
exclude_query = input("Слова для исключения (Enter для пропуска): ").strip()

# Автоматическое установление актуальных дат
current_date = datetime.now()
date_to = current_date.strftime("%Y-%m-%d")
date_from = (current_date - timedelta(days=5)).strftime("%Y-%m-%d")

print(f"\n📅 Автоматически установлены актуальные даты:")
print(f"   - Дата публикации ОТ: {date_from} (5 дней назад)")
print(f"   - Дата публикации ДО: {date_to} (сегодня)")

# Опционально: Ручной ввод дат если нужно
change_dates = input("\nХотите изменить даты поиска? (y/N): ").strip().lower()
if change_dates == 'y':
    date_from = input("Введите дату ОТ (ГГГГ-ММ-ДД): ").strip()
    date_to = input("Введите дату ДО (ГГГГ-ММ-ДД): ").strip()
    print(f"✅ Установлены ручные даты: {date_from} - {date_to}")

# Дополнительно: Дата окончания подачи заявок (опционально)
use_deadline = input("\nХотите установить даты окончания подачи заявок? (y/N): ").strip().lower()
if use_deadline == 'y':
    deadline_from = input("Введите дату окончания подачи ОТ (ГГГГ-ММ-ДД): ").strip()
    deadline_to = input("Введите дату окончания подачи ДО (ГГГГ-ММ-ДД): ").strip()
else:
    deadline_from = None
    deadline_to = None

# Ценовые параметры
price_from = input("Цена ОТ (Enter для пропуска): ").strip()
price_to = input("Цена ДО (Enter для пропуска): ").strip()

max_tenders = input("Максимальное количество тендеров для анализа (по умолчанию 50, максимум 500): ").strip()
max_tenders = int(max_tenders) if max_tenders.isdigit() else 50
max_tenders = min(max_tenders, 500)  # Ограничиваем максимум 500 тендерами

# Формируем правильные параметры поиска согласно документации API
search_params = {
    "page": 1,
    "per_page": min(100, max_tenders),

    # Правильные названия параметров из документации
    "SearchString": search_query if search_query else None,
    "ExcludeSearchString": exclude_query if exclude_query else None,

    # Правильные параметры дат публикации
    "PublishDateFrom": date_from,
    "PublishDateTo": date_to,

    # Правильные параметры дат окончания подачи заявок (если указаны)
    "RequestEndDateFrom": deadline_from if deadline_from else None,
    "RequestEndDateTo": deadline_to if deadline_to else None,

    # Правильные параметры цены
    "InitialPriceFrom": float(price_from) if price_from and price_from.replace('.', '').isdigit() else None,
    "InitialPriceTo": float(price_to) if price_to and price_to.replace('.', '').isdigit() else None
}

# Очищаем параметры от None значений
clean_params = {k: v for k, v in search_params.items() if v is not None}

print(f"\n🚀 ПАРАМЕТРЫ ПОИСКА:")
print(f"   - Максимальное количество тендеров: {max_tenders}")
print(f"   - Ключевые слова: {search_query if search_query else 'не указаны'}")
print(f"   - Исключаемые слова: {exclude_query if exclude_query else 'не указаны'}")
print(f"   - Дата публикации ОТ: {date_from}")
print(f"   - Дата публикации ДО: {date_to}")
if deadline_from and deadline_to:
    print(f"   - Дата окончания подачи ОТ: {deadline_from}")
    print(f"   - Дата окончания подачи ДО: {deadline_to}")
print(f"   - Цена ОТ: {price_from if price_from else 'не указана'}")
print(f"   - Цена ДО: {price_to if price_to else 'не указана'}")
print(f"   - Источник описания компании: {company_file_url}")

if max_tenders > 100:
    print(f"   ⚠️  Будет выполнено несколько запросов для получения {max_tenders} тендеров")

# Запуск основного процесса
print("\n🚀 ЗАПУСК ОСНОВНОГО ПРОЦЕССА ОБРАБОТКИ ТЕНДЕРОВ")
print("=" * 80)

# Исправленная инициализация пайплайна
pipeline = TenderProcessingPipeline(client, yandex_gpt_api_key)

results = pipeline.process(
    company_file_url=company_file_url,
    search_params=clean_params,
    max_tenders=max_tenders
)

print(f"\n📊 ИТОГИ ОБРАБОТКИ:")
print(f"  - Статус: {results['status']}")
print(f"  - Найдено тендеров: {results['total_tenders_found']}")
print(f"  - Релевантных: {results['relevant_tenders']}")
print(f"  - Обработано: {results['processed_tenders']}")
print(f"  - Сгенерировано ключевых слов: {results['keywords_generated']}")
print(f"  - Время обработки: {results['processing_time']:.2f} сек.")
print(f"  - Результаты: {results['output_file']}")
print(f"  - Ошибок: {len(results['errors'])}")

if results['status'] == 'success':
    print("\n🎉 ПРОЦЕСС ОБРАБОТКИ ЗАВЕРШЕН УСПЕШНО!")
    print("Файл с результатами доступен для скачивания:")
    files.download(results['output_file'])
else:
    print("\n❌ ПРОИЗОШЛА ОШИБКА В ПРОЦЕССЕ ОБРАБОТКИ")
    for error in results['errors']:
        print(f"  - {error}")


🎯 НАСТРОЙКА ПАРАМЕТРОВ ПОИСКА
Введите ссылку на файл с описанием компании (DOCX/PDF/TXT/Google Docs): https://docs.google.com/document/d/1RggP8wiq6zIW0Wwx9qbHUCJOny7z13NmxSRmSdvwBs8/edit?usp=sharing

📋 ДОПОЛНИТЕЛЬНЫЕ ПАРАМЕТРЫ ПОИСКА:
Основные ключевые слова для поиска (например: вал шестерня оборудование): по чертежам вал шестерня привод ролик редуктор
Слова для исключения (Enter для пропуска): авто трактор сельхозтехника ВАЗ МАЗ КАМАЗ

📅 Автоматически установлены актуальные даты:
   - Дата публикации ОТ: 2025-10-24 (5 дней назад)
   - Дата публикации ДО: 2025-10-29 (сегодня)

Хотите изменить даты поиска? (y/N): n

Хотите установить даты окончания подачи заявок? (y/N): n
Цена ОТ (Enter для пропуска): 
Цена ДО (Enter для пропуска): 
Максимальное количество тендеров для анализа (по умолчанию 50, максимум 500): 

🚀 ПАРАМЕТРЫ ПОИСКА:
   - Максимальное количество тендеров: 50
   - Ключевые слова: по чертежам вал шестерня привод ролик редуктор
   - Исключаемые слова: авто трактор сельхозте

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>